# 离线回测指标策略

本notebook展示了如何使用backtrader进行币安交易策略的回测，包含了SMA和RSI指标的使用。

In [ ]:
import datetime as dt
import backtrader as bt
from backtrader_binance import BinanceStore
from ConfigBinance.Config import Config  # Configuration file

## 自定义指标
创建一个简单的指标来判断一个数据是否低于另一个数据

In [ ]:
class UnderOver(bt.Indicator):
    lines = ('underover',)
    params = dict(data2=20)
    plotinfo = dict(plot=True)

    def __init__(self):
        self.l.underover = self.data < self.p.data2

## 交易策略
实现一个基于SMA和RSI的交易策略

In [ ]:
class SimpleSMAStrategy(bt.Strategy):
    """ Backtest strategy demonstration with SMA indicators """
    params = (  # Parameters of the trading system
        ('coin_target', ''),
        ('timeframe', ''),
        ('leverage', ''),
    )

    def __init__(self):
        """Initialization, adding indicators for each ticker"""
        self.orders = {}  # All orders as a dict, for this particularly trading strategy one ticker is one order
        for d in self.datas:  # Running through all the tickers
            self.orders[d._name] = None  # There is no order for ticker yet

        # creating indicators for each ticker
        self.sma1 = {}
        self.sma2 = {}
        self.sma3 = {}
        self.crossover = {}
        self.underover_sma = {}

        for i in range(len(self.datas)):
            ticker = list(self.dnames.keys())[i]    # key name is ticker name
            self.sma1[ticker] = bt.indicators.SMA(self.datas[i], period=9)  # SMA1 indicator
            self.sma2[ticker] = bt.indicators.SMA(self.datas[i], period=30)  # SMA2 indicator
            self.sma3[ticker] = bt.indicators.SMA(self.datas[i], period=60)  # SMA3 indicator

            # signal 1 - intersection of a fast SMA from bottom to top of a slow SMA
            self.crossover[ticker] = bt.ind.CrossOver(self.sma1[ticker], self.sma2[ticker])  # crossover SMA1 and SMA2

            # signal 2 - when SMA3 is below SMA2
            self.underover_sma[ticker] = UnderOver(self.sma3[ticker].lines.sma, data2=self.sma2[ticker].lines.sma)


    def next(self):
        """Arrival of a new ticker candle"""
        for data in self.datas:  # Running through all the requested bars of all tickers
            ticker = data._name
            status = data._state  # 0 - Live data, 1 - History data, 2 - None
            _interval = self.p.timeframe

            if status in [0, 1]:
                if status: _state = "False - History data"
                else: _state = "True - Live data"

                print('{} / {} [{}] - Open: {}, High: {}, Low: {}, Close: {}, Volume: {} - Live: {}'.format(bt.num2date(data.datetime[0]), ticker, _interval, data.open[0], data.high[0], data.low[0], data.close[0], data.volume[0], _state, ))

                # signals to open position
                signal1 = self.crossover[ticker].lines.crossover[0]  # signal 1 - intersection of a fast SMA from bottom to top of a slow SMA
                signal2 = self.underover_sma[ticker]  # signal 2 - when SMA3 is below SMA2

                if not self.getposition(data):  # If there is no position
                    if signal1 == 1:
                        if signal2 == 1:
                            # buy
                            free_money = self.broker.getcash()
                            price = data.close[0]  # by closing price
                            size = (free_money * self.p.leverage / price) * 0.10  # 10% of available funds * leverage

                            print("-"*50)
                            print(f"\t - Let's buy {ticker}, size = {size} at price = {price}, depo: {self.broker.getcash()} {self.p.coin_target}")
                            self.orders[ticker] = self.buy(data=data, exectype=bt.Order.Limit, price=price, size=size)
                            print(f"\t - Order has been submitted to buy {ticker}, size={size}, price={price}")
                            print("-" * 50)

                else:  # If there is a position
                    if signal1 == 0 and signal2 == 0:
                        # sell
                        if self.orders[ticker]:
                            print("-" * 50)
                            print(f"\t - Let's sell by market {ticker}... size:{self.orders[ticker].size}")
                            self.orders[ticker] = self.close(data=data)  # Request to close a position at the market price
                            print(f"\t - Order has been submitted to sell by market {ticker}")
                            print("-" * 50)

    def notify_order(self, order):
        """Changing the status of the order"""
        ticker = order.data._name  # Name of ticker from order
        self.log(f'Order number {order.ref} {order.info["order_number"]} {order.getstatusname()} {"Buy" if order.isbuy() else "Sell"} {ticker} {order.size} @ {order.price}')
        if order.status == bt.Order.Completed:  # If the order is fully executed
            if order.isbuy():  # The order to buy
                self.log(f'\t*** BUY ORDER COMPLETED for {ticker} price: {order.executed.price:.2f}, size = {order.size}, cost: {order.executed.value:.2f}, comm: {order.executed.comm:.2f}, depo={self.cerebro.broker.getcash():.2f} {self.p.coin_target}')
            else:  # The order to sell
                self.log(f'\t*** SELL ORDER COMPLETED for {ticker} price: {order.executed.price:.2f}, size = {order.size}, cost: {order.executed.value:.2f}, comm: {order.executed.comm:.2f}, depo={self.cerebro.broker.getcash():.2f} {self.p.coin_target}')
                self.orders[ticker] = None  # Reset the order to enter the position

    def notify_trade(self, trade):
        """Changing the position status"""
        if trade.isclosed:  # If the position is closed
            self.log(f'Profit on a closed position {trade.getdataname()} Total={trade.pnl:.2f}, commission={abs(trade.pnl-trade.pnlcomm):.2f}')

    def log(self, txt, dt=None):
        """Print string with date to the console"""
        dt = bt.num2date(self.datas[0].datetime[0]) if dt is None else dt  # date or date of the current bar
        print(f'{dt.strftime("%Y-%m-%d %H:%M:%S")} {txt}')  # Print the date and time with the specified text to the console

## 运行回测
设置回测参数并执行回测

In [ ]:
if __name__ == '__main__':
    cerebro = bt.Cerebro(quicknotify=True)

    cerebro.broker.setcash(2000)
    cerebro.broker.setcommission(commission=0.0015)

    coin_target = 'USDT'
    symbol = 'BTC' + coin_target
    symbol2 = 'ETH' + coin_target

    store = BinanceStore(
        api_key=Config.BINANCE_API_KEY,
        api_secret=Config.BINANCE_API_SECRET,
        coin_target=coin_target,
        testnet=False)

    timeframe = "D1"
    from_date = dt.datetime.utcnow() - dt.timedelta(days=365*3)
    data = store.getdata(timeframe=bt.TimeFrame.Days, compression=1, dataname=symbol, start_date=from_date, LiveBars=False)
    data2 = store.getdata(timeframe=bt.TimeFrame.Days, compression=1, dataname=symbol2, start_date=from_date, LiveBars=False)

    cerebro.adddata(data)
    cerebro.adddata(data2)

    cerebro.addstrategy(RSIStrategy, coin_target=coin_target, timeframe=timeframe)

    cerebro.run()
    cerebro.plot()

    print()
    print("$"*77)
    print(f"Liquidation value of the portfolio: {cerebro.broker.getvalue()}")
    print(f"Remaining available funds: {cerebro.broker.getcash()}")
    print("$" * 77)